<a href="https://colab.research.google.com/github/EdenGebremedhin/AgriMargin_Kaggle_Capstone_Project/blob/main/AgriMargin_Kaggle_Capstone_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AgriMargin — AI Agents Capstone Submission
### Kaggle: *5-Day AI Agents — Intensive Vibe Coding Capstone Project*
**Track: Agents for Good**

An AI decision-agent that helps smallholder farmers decide whether treating
a crop problem is economically worth it, combining pest/disease diagnosis,
climate-risk assessment, and market price into one recommendation, and
(with explicit human confirmation) acting on it.



## 1. Setup

In [109]:
# Install Google's Agent Development Kit
!pip install -q google-adk


In [110]:
import os
from getpass import getpass

# Get a free key at https://aistudio.google.com/apikey
if not os.environ.get("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = getpass("Please enter your Google API key: ")
print("API key set." if os.environ.get("GOOGLE_API_KEY") else "No key set yet.")

API key set.


## 2. Tools

Two kinds of functions below, and it matters which is which:

- **Real logic** — `compute_disease_risk()` and `compute_economics()` are
  genuine, deterministic calculations. This is where the actual
  "intelligence" of the economic-decision layer lives.
- **Stub / TODO** — `get_weather_forecast()` and `get_market_price()`
  return plausible-shaped fake data so the whole pipeline is runnable
  end-to-end right now. They are clearly labeled `STUB DATA` in their
  return values — replace them with real APIs before this touches a real
  farmer.


In [111]:
%%writefile tools.py
"""
tools.py — Function tools used by the AgriMargin agents.

Two kinds of functions live here, and it matters which is which:

1. REAL LOGIC — compute_disease_risk() and compute_economics() are genuine,
   deterministic calculations. They don't call any external API and will
   give you real numbers today. This is where the actual "intelligence"
   of the economic-decision layer lives.

2. STUB / TODO — get_weather_forecast() and get_market_price() return
   plausible-shaped fake data so the whole pipeline is runnable end to end
   right now. Before this goes anywhere near a real farmer, swap these for
   real calls (a weather API, a market-price API/MCP server). They are
   clearly marked below — do not mistake their output for real data.

send_dealer_order() is wrapped with require_confirmation=True where it's
registered as a tool (see agents.py) — the agent cannot execute it without
an explicit human "yes" being returned to the tool_confirmation flow. This
is the safety rail the capstone rubric wants under "security features".
"""

from datetime import datetime, timedelta
import random


# ---------------------------------------------------------------------------
# 1. WEATHER — STUB. Replace with a real weather API (e.g. OpenWeatherMap,
#    or a national meteorological service API) before production use.
# ---------------------------------------------------------------------------
def get_weather_forecast(location: str) -> dict:
    """Get a short-term weather forecast for a farm location.

    Args:
        location: Town/district name or lat,lon string.

    Returns:
        dict with avg_temp_c, avg_humidity_pct, rain_expected_mm over the
        next 5 days.
    """
    # STUB: deterministic-but-fake numbers seeded by location name so demo
    # runs are repeatable. Replace this whole function body with a real
    # API call. Do not treat this output as real weather.
    seed = sum(ord(c) for c in location)
    rnd = random.Random(seed)
    return {
        "location": location,
        "forecast_days": 5,
        "avg_temp_c": round(rnd.uniform(22, 34), 1),
        "avg_humidity_pct": round(rnd.uniform(45, 90), 1),
        "rain_expected_mm": round(rnd.uniform(0, 40), 1),
        "source": "STUB DATA — replace with real weather API",
    }


# ---------------------------------------------------------------------------
# 2. DISEASE RISK — REAL LOGIC. A simplified but genuine heuristic: fungal
#    and bacterial crop disease spread risk rises with humidity and warm
#    temperatures. This is a simplified version of logic used in real
#    agronomy decision-support tools (e.g. blight risk models), not a
#    black box. Tune the thresholds per-crop with an agronomist before
#    relying on it for real decisions.
# ---------------------------------------------------------------------------
def compute_disease_risk(avg_temp_c: float, avg_humidity_pct: float, crop: str) -> dict:
    """Compute a 0-100 disease-spread risk index from weather conditions.

    Args:
        avg_temp_c: Average forecast temperature in Celsius.
        avg_humidity_pct: Average forecast relative humidity in percent.
        crop: Crop name (e.g. "tomato", "potato", "maize").

    Returns:
        dict with risk_score (0-100), risk_level, and rationale.
    """
    # Fungal pathogens (early blight, late blight, powdery mildew, etc.)
    # generally thrive at humidity > 80% and moderate-warm temps 20-28C.
    humidity_component = max(0.0, (avg_humidity_pct - 50)) * 1.4
    if 18 <= avg_temp_c <= 28:
        temp_component = 30
    elif 28 < avg_temp_c <= 34:
        temp_component = 18
    else:
        temp_component = 8

    risk_score = min(100, round(humidity_component * 0.6 + temp_component))

    if risk_score >= 70:
        level = "high"
    elif risk_score >= 40:
        level = "moderate"
    else:
        level = "low"

    return {
        "crop": crop,
        "risk_score": risk_score,
        "risk_level": level,
        "rationale": (
            f"At {avg_humidity_pct}% humidity and {avg_temp_c}C, conditions are "
            f"{'favorable' if risk_score >= 40 else 'not strongly favorable'} "
            f"for fungal disease spread on {crop}."
        ),
    }


# ---------------------------------------------------------------------------
# 3. MARKET PRICES — STUB. Replace with a real mandi/market-price API
#    (e.g. a government open-data ag-market API for your region) or an
#    MCP server exposing that data. See README for how to swap this for
#    an MCPToolset connection.
# ---------------------------------------------------------------------------
def get_market_price(crop: str, market_name: str) -> dict:
    """Get current and recent market price trend for a crop at a market.

    Args:
        crop: Crop name.
        market_name: Local market/mandi name.

    Returns:
        dict with price_per_kg, currency, trend_pct_7d.
    """
    # STUB: replace with a real market-price API call. Numbers below are
    # illustrative only.
    seed = sum(ord(c) for c in crop + market_name)
    rnd = random.Random(seed)
    price = round(rnd.uniform(8, 40), 2)
    trend = round(rnd.uniform(-15, 15), 1)
    return {
        "crop": crop,
        "market": market_name,
        "price_per_kg": price,
        "currency": "local_unit",
        "trend_pct_7d": trend,
        "as_of": datetime.now().strftime("%Y-%m-%d"),
        "source": "STUB DATA — replace with real market price API/MCP server",
    }


# ---------------------------------------------------------------------------
# 4. ECONOMIC DECISION — REAL LOGIC. Compares cost of treating the crop
#    against the expected value of yield saved. This is the core novel
#    piece: it's the calculation existing diagnosis-only or price-only
#    apps don't do.
# ---------------------------------------------------------------------------
def compute_economics(
    treatment_cost_per_hectare: float,
    expected_yield_loss_pct_if_untreated: float,
    expected_yield_kg_per_hectare: float,
    price_per_kg: float,
    days_to_harvest: int,
) -> dict:
    """Decide whether treating the crop is economically worthwhile.

    Args:
        treatment_cost_per_hectare: Cost of the fungicide/pesticide + labor.
        expected_yield_loss_pct_if_untreated: Estimated % yield lost if the
            disease/pest is left untreated (agronomist estimate or lookup
            table by disease severity).
        expected_yield_kg_per_hectare: Expected total yield per hectare.
        price_per_kg: Current market price per kg.
        days_to_harvest: Days remaining until harvest.

    Returns:
        dict with recommendation, expected_loss_value, net_benefit.
    """
    expected_loss_kg = expected_yield_kg_per_hectare * (
        expected_yield_loss_pct_if_untreated / 100
    )
    expected_loss_value = round(expected_loss_kg * price_per_kg, 2)
    net_benefit = round(expected_loss_value - treatment_cost_per_hectare, 2)

    if days_to_harvest <= 7:
        # Too close to harvest — treatment may not have time to help, and
        # pre-harvest interval (PHI) restrictions likely apply.
        recommendation = "do_not_spray_too_close_to_harvest"
        reason = (
            f"Only {days_to_harvest} days to harvest — check the pre-harvest "
            "interval on the product label before spraying."
        )
    elif net_benefit > treatment_cost_per_hectare * 0.25:
        recommendation = "spray"
        reason = (
            f"Expected loss value ({expected_loss_value}) clearly exceeds "
            f"treatment cost ({treatment_cost_per_hectare}). Net benefit: "
            f"{net_benefit}."
        )
    elif net_benefit > 0:
        recommendation = "spray_marginal"
        reason = (
            "Treating is worthwhile but the margin is thin "
            f"(net benefit: {net_benefit}). Consider a cheaper input option."
        )
    else:
        recommendation = "do_not_spray"
        reason = (
            f"Treatment cost ({treatment_cost_per_hectare}) exceeds expected "
            f"loss value ({expected_loss_value}). Not economically justified."
        )

    return {
        "recommendation": recommendation,
        "reason": reason,
        "expected_loss_value": expected_loss_value,
        "treatment_cost": treatment_cost_per_hectare,
        "net_benefit": net_benefit,
    }


# ---------------------------------------------------------------------------
# 5. ACTIONS — draft is safe/no side effects. send_dealer_order and
#    schedule_reminder actually do something and must require confirmation.
#    The require_confirmation=True flag is set in agents.py when these are
#    registered as tools, not here — this file only defines behavior.
# ---------------------------------------------------------------------------
def draft_dealer_order(crop: str, product_name: str, quantity_kg: float, farmer_name: str) -> dict:
    """Draft (but do not send) a purchase order message to a local input dealer."""
    message = (
        f"Order request from {farmer_name}: {quantity_kg}kg of {product_name} "
        f"for {crop} treatment. Please confirm availability and price."
    )
    return {"draft_message": message, "status": "drafted_not_sent"}


def send_dealer_order(dealer_contact: str, order_message: str) -> dict:
    """Actually send the order message to the dealer (e.g. via WhatsApp Business API).

    This is a side-effecting action and MUST require human confirmation
    before it runs (enforced via require_confirmation=True in agents.py).
    """
    # STUB: replace with a real WhatsApp Business Cloud API / Twilio call.
    return {
        "status": "sent (SIMULATED — no real message was sent)",
        "dealer_contact": dealer_contact,
        "message": order_message,
        "sent_at": datetime.now().isoformat(),
    }


def schedule_reapplication_reminder(crop: str, days_from_now: int) -> dict:
    """Schedule a reminder to re-check/reapply treatment."""
    reminder_date = (datetime.now() + timedelta(days=days_from_now)).strftime("%Y-%m-%d")
    return {
        "crop": crop,
        "reminder_date": reminder_date,
        "status": "scheduled (SIMULATED — hook up to a real calendar/SMS reminder service)",
    }


Overwriting tools.py


In [112]:
# Quick sanity check of the real logic (no API key needed for this part)
from tools import get_weather_forecast, compute_disease_risk, get_market_price, compute_economics

w = get_weather_forecast("Jimma")
r = compute_disease_risk(w["avg_temp_c"], w["avg_humidity_pct"], "tomato")
p = get_market_price("tomato", "Jimma Market")
e = compute_economics(
    treatment_cost_per_hectare=1800,
    expected_yield_loss_pct_if_untreated=25,
    expected_yield_kg_per_hectare=20000,
    price_per_kg=p["price_per_kg"],
    days_to_harvest=20,
)
print("Weather:", w)
print("Disease risk:", r)
print("Market price:", p)
print("Economic decision:", e)


Weather: {'location': 'Jimma', 'forecast_days': 5, 'avg_temp_c': 29.5, 'avg_humidity_pct': 76.0, 'rain_expected_mm': 25.5, 'source': 'STUB DATA — replace with real weather API'}
Disease risk: {'crop': 'tomato', 'risk_score': 40, 'risk_level': 'moderate', 'rationale': 'At 76.0% humidity and 29.5C, conditions are favorable for fungal disease spread on tomato.'}
Market price: {'crop': 'tomato', 'market': 'Jimma Market', 'price_per_kg': 23.36, 'currency': 'local_unit', 'trend_pct_7d': 3.2, 'as_of': '2026-07-06', 'source': 'STUB DATA — replace with real market price API/MCP server'}
Economic decision: {'recommendation': 'spray', 'reason': 'Expected loss value (116800.0) clearly exceeds treatment cost (1800). Net benefit: 115000.0.', 'expected_loss_value': 116800.0, 'treatment_cost': 1800, 'net_benefit': 115000.0}


## 3. Agents

Five specialist agents behind one orchestrator, built on the real
`google-adk` API (`Agent` + `sub_agents` delegation).


In [113]:
%%writefile agents.py
"""
agents.py — The AgriMargin multi-agent system, built on real google-adk
(pip install google-adk, version 2.x — verified against the installed
package, not guessed).

Pattern used: a root "orchestrator" Agent with sub_agents. ADK's LlmAgent
automatically exposes each sub-agent as something the parent can delegate
to mid-conversation (transfer_to_agent), so the orchestrator can reason
about a farmer's message, call the right specialists in order, and
combine their outputs — without you hand-writing the control flow.

Model: all agents use "gemini-2.5-flash" by default — fast and cheap
enough for a conversational agronomy assistant. Swap to a stronger model
for the Diagnosis Agent if photo-ID accuracy needs improving.
"""

import os
from google.adk import Agent
from google.adk.tools import FunctionTool

from tools import (
    get_weather_forecast,
    compute_disease_risk,
    get_market_price,
    compute_economics,
    draft_dealer_order,
    send_dealer_order,
    schedule_reapplication_reminder,
)

MODEL = os.environ.get("AGRIMARGIN_MODEL", "gemini-2.5-flash")


# ---------------------------------------------------------------------------
# 1. Diagnosis Agent
# ---------------------------------------------------------------------------
# In production this agent receives the farmer's photo directly as a
# multimodal message part (Gemini's native vision, not a function tool) —
# ADK passes images straight to the model. For text-only testing (no
# photo available), it reasons from a symptom description instead.
diagnosis_agent = Agent(
    name="diagnosis_agent",
    model=MODEL,
    description=(
        "Identifies crop pest/disease from a photo or a text description of "
        "symptoms, and reports a confidence level."
    ),
    instruction=(
        "You are a plant pathology expert. You will be given either an image "
        "of a crop leaf/plant, or a text description of symptoms. Identify "
        "the most likely pest or disease. Respond with: the crop, the "
        "diagnosis, your confidence (low/medium/high), and one sentence of "
        "reasoning. If the input is too ambiguous to diagnose confidently, "
        "say so plainly instead of guessing — do not invent a diagnosis you "
        "are not reasonably confident in."
    ),
)


# ---------------------------------------------------------------------------
# 2. Weather-Risk Agent
# ---------------------------------------------------------------------------
weather_risk_agent = Agent(
    name="weather_risk_agent",
    model=MODEL,
    description="Fetches forecast weather and computes disease-spread risk for a crop.",
    instruction=(
        "You assess disease-spread risk from weather. Given a location and "
        "crop, call get_weather_forecast to get the forecast, then call "
        "compute_disease_risk with those numbers. Report the risk_level and "
        "rationale clearly. Do not skip calling the tools — do not estimate "
        "weather from general knowledge."
    ),
    tools=[
        FunctionTool(get_weather_forecast),
        FunctionTool(compute_disease_risk),
    ],
)


# ---------------------------------------------------------------------------
# 3. Market-Intelligence Agent
# ---------------------------------------------------------------------------
# NOTE: get_market_price is a local stub function right now. To satisfy the
# capstone's "MCP server" requirement for real, point this agent at an
# MCPToolset instead — see README.md "Swapping in a real MCP server".
market_intel_agent = Agent(
    name="market_intel_agent",
    model=MODEL,
    description="Fetches current market/mandi price and short-term trend for a crop.",
    instruction=(
        "You report crop market prices. Given a crop and market name, call "
        "get_market_price and report the price, currency, and 7-day trend. "
        "Always call the tool — never guess a price from memory."
    ),
    tools=[FunctionTool(get_market_price)],
)


# ---------------------------------------------------------------------------
# 4. Economic-Advisor Agent
# ---------------------------------------------------------------------------
economic_advisor_agent = Agent(
    name="economic_advisor_agent",
    model=MODEL,
    description=(
        "Combines diagnosis, disease risk, and market price into a spray / "
        "wait / sell recommendation with the underlying cost-benefit math."
    ),
    instruction=(
        "You are the decision-making core of this system. You will be given "
        "a diagnosis, a disease risk level, and a market price, plus farm "
        "figures (treatment cost, expected yield, days to harvest). Call "
        "compute_economics with your best estimates for "
        "expected_yield_loss_pct_if_untreated based on the diagnosis "
        "confidence and risk level (low risk+low confidence diagnosis = "
        "lower estimate, e.g. 5-10%; high risk+high confidence = higher, "
        "e.g. 20-35%). State your recommendation plainly, show the numbers, "
        "and explain the reasoning in one paragraph a farmer can actually "
        "understand — no jargon. If any critical figure is missing (crop "
        "price, yield estimate, treatment cost), ask for it instead of "
        "guessing."
    ),
    tools=[FunctionTool(compute_economics)],
)


# ---------------------------------------------------------------------------
# 5. Action Agent — the only agent allowed to touch anything external.
# ---------------------------------------------------------------------------
# send_dealer_order is registered with require_confirmation=True. ADK will
# pause and require an explicit human confirmation before this tool
# actually executes — this is the safety rail. draft_dealer_order has no
# side effects, so it does not need confirmation.
action_agent = Agent(
    name="action_agent",
    model=MODEL,
    description=(
        "Drafts and (with explicit farmer confirmation) sends a dealer "
        "order, and schedules a reapplication reminder. Never acts without "
        "confirmation."
    ),
    instruction=(
        "You take action ONLY after the economic_advisor_agent has "
        "recommended spraying. First call draft_dealer_order to prepare the "
        "order message and show it to the farmer. Only call send_dealer_order "
        "after the farmer has explicitly approved the draft — never send "
        "without approval, and never treat silence as approval. After a "
        "successful send, call schedule_reapplication_reminder for 10-14 "
        "days out. If the economic recommendation was not to spray, do not "
        "draft or send any order — just confirm to the farmer that no "
        "action is needed."
    ),
    tools=[
        FunctionTool(draft_dealer_order),
        FunctionTool(send_dealer_order, require_confirmation=True),
        FunctionTool(schedule_reapplication_reminder, require_confirmation=True),
    ],
)


# ---------------------------------------------------------------------------
# Orchestrator — root agent, delegates to the five specialists above.
# ---------------------------------------------------------------------------
orchestrator = Agent(
    name="agrimargin_orchestrator",
    model=MODEL,
    description=(
        "AgriMargin: helps a farmer decide whether to spray, wait, or sell, "
        "and can act on that decision."
    ),
    instruction=(
        "You are AgriMargin, an assistant that helps a smallholder farmer "
        "make an economically sound decision about a crop problem, then "
        "optionally acts on it. For a new crop-problem query, work through "
        "these steps in order, delegating to the right specialist agent for "
        "each: \n"
        "1. diagnosis_agent — identify the pest/disease.\n"
        "2. weather_risk_agent — assess disease-spread risk for the farmer's "
        "location and crop.\n"
        "3. market_intel_agent — get the current price for the crop at the "
        "farmer's local market.\n"
        "4. economic_advisor_agent — combine all of the above into a spray/"
        "wait/sell recommendation, asking the farmer for any missing farm "
        "figures (treatment cost, expected yield per hectare, days to "
        "harvest) if needed.\n"
        "5. Only if the recommendation is to spray AND the farmer wants "
        "help acting on it, delegate to action_agent.\n"
        "Keep your own messages to the farmer short, plain-language, and "
        "in the language they're writing in. Never skip the economic step "
        "and jump straight to 'yes, spray' just because a disease was "
        "identified — the whole point of this system is that diagnosis "
        "alone is not a recommendation."
    ),
    sub_agents=[
        diagnosis_agent,
        weather_risk_agent,
        market_intel_agent,
        economic_advisor_agent,
        action_agent,
    ],
)


root_agent = orchestrator  # ADK convention: root_agent is the entrypoint.


Overwriting agents.py


In [114]:
from agents import orchestrator, root_agent

print("Orchestrator:", orchestrator.name)
print("Sub-agents:", [a.name for a in orchestrator.sub_agents])
for a in orchestrator.sub_agents:
    print(" ", a.name, "-> tools:", [t.name if hasattr(t, "name") else t.__name__ for t in a.tools])


Orchestrator: agrimargin_orchestrator
Sub-agents: ['diagnosis_agent', 'weather_risk_agent', 'market_intel_agent', 'economic_advisor_agent', 'action_agent']
  diagnosis_agent -> tools: []
  weather_risk_agent -> tools: ['get_weather_forecast', 'compute_disease_risk']
  market_intel_agent -> tools: ['get_market_price']
  economic_advisor_agent -> tools: ['compute_economics']
  action_agent -> tools: ['draft_dealer_order', 'send_dealer_order', 'schedule_reapplication_reminder']


## 4. Run AgriMargin

In [115]:
import asyncio
from google.adk import Runner
from google.adk.sessions import InMemorySessionService
from agents import root_agent

APP_NAME, USER_ID, SESSION_ID = "agrimargin", "demo_farmer", "demo_session"

DEMO_MESSAGE = (
    "My tomato leaves have dark concentric rings on the lower leaves and "
    "some yellowing around them. Weather has been humid, I'm near Jimma. "
    "Should I spray? Treatment costs about 1800 per hectare, I expect "
    "about 20000kg/hectare yield, and harvest is 20 days away. Market is "
    "the Jimma Market."
)

async def run_demo():
    session_service = InMemorySessionService()
    await session_service.create_session(app_name=APP_NAME, user_id=USER_ID, session_id=SESSION_ID)
    runner = Runner(agent=root_agent, app_name=APP_NAME, session_service=session_service)
    runner.run_debug(DEMO_MESSAGE, user_id=USER_ID, session_id=SESSION_ID)

await run_demo()  # Colab supports top-level await

/tmp/ipykernel_13412/103748343.py:20: RuntimeWarning: coroutine 'Runner.run_debug' was never awaited
  runner.run_debug(DEMO_MESSAGE, user_id=USER_ID, session_id=SESSION_ID)


In [116]:
# Optional: ask your own question
USER_MESSAGE = "Is it worth spraying my maize for fall armyworm with 10 days to harvest?"

async def run_custom():
    session_service = InMemorySessionService()
    await session_service.create_session(app_name=APP_NAME, user_id=USER_ID, session_id="custom_session")
    runner = Runner(agent=root_agent, app_name=APP_NAME, session_service=session_service)
    runner.run_debug(USER_MESSAGE, user_id=USER_ID, session_id="custom_session")

await run_custom()

/tmp/ipykernel_13412/3564041740.py:8: RuntimeWarning: coroutine 'Runner.run_debug' was never awaited
  runner.run_debug(USER_MESSAGE, user_id=USER_ID, session_id="custom_session")


## 5. Architecture

```
agrimargin_orchestrator (root Agent)
 ├── diagnosis_agent        — pest/disease ID from photo or symptoms
 ├── weather_risk_agent     — forecast → disease-spread risk index
 ├── market_intel_agent     — current crop price + trend
 ├── economic_advisor_agent — spray / wait / sell recommendation + math
 └── action_agent           — drafts + (confirmed) sends dealer order,
                               schedules reapplication reminder
```

The orchestrator is a standard ADK `Agent` with `sub_agents=[...]` - it
delegates to each specialist automatically based on the conversation
(ADK's built-in `transfer_to_agent`), rather than a hand-coded pipeline.

**Security feature:** `send_dealer_order` and
`schedule_reapplication_reminder` are registered with
`require_confirmation=True` in `agents.py` - ADK will not execute either
without an explicit human "yes". The agent cannot place an order or send
a message unattended.



## 6. Kaggle Writeup
# AgriMargin: The Missing Economic Layer for Smallholder Crop Decisions

### Subtitle: A multi-agent system that doesn't just diagnose crop disease — it decides whether treating it is worth the money, then safely acts on that decision.

**Track: Agents for Good** (agriculture) — the underlying decision logic and
dealer-order flow also transfer directly to Agents for Business if a
cooperative or agri-input company deploys it.

---

## The problem

A smallholder farmer with a sick crop today has to stitch together
information from several disconnected places: a photo-ID app tells them
what disease it might be, a weather app or their own intuition tells them
if conditions favor its spread, and a separate market app or a trip to the
mandi tells them the current price. Nothing puts those three pieces of
information together and answers the question that actually determines
their income: **is treating this worth the money, given how close I am to
harvest and what the crop will sell for?**

That gap has a real cost. Farmers either over-spend on treatment that
doesn't pay for itself, or under-treat a genuinely high-risk outbreak
because they never ran the numbers. Existing tools (Plantix-style photo
diagnosis apps, government mandi-price apps, generic weather apps) each
solve one piece of this and stop. None of them close the loop into a
recommendation and an action.

## Why agents, specifically

This isn't a single-model classification problem — it's a small
*research-and-decide* workflow: gather evidence from three different
domains (plant pathology, meteorology, market economics), reason over all
of it together, and only then, optionally, take a real-world action
(placing an order) that has to be gated by human approval. That's exactly
the shape of problem multi-agent orchestration is for: specialist agents
that each do one job well, coordinated by an orchestrator that knows the
right sequence and can hand off between them mid-conversation, with a
tool-execution layer that can be paused for confirmation before anything
irreversible happens.

## Solution & architecture

AgriMargin is built on Google's Agent Development Kit (ADK) as five
specialist agents behind one orchestrator:

1. **Diagnosis Agent** — identifies the pest/disease from a farmer's photo
   (or symptom description in text-only testing) using Gemini's native
   multimodal reasoning, and reports a confidence level rather than a
   false-certain answer.
2. **Weather-Risk Agent** — pulls a short-term forecast and computes a
   disease-spread risk index from temperature and humidity. The risk
   heuristic (higher humidity + moderate-warm temperatures favor fungal
   spread) is real, working logic, not a placeholder.
3. **Market-Intelligence Agent** — fetches the current price and 7-day
   trend for the crop at the farmer's local market. This is the natural
   slot for an MCP server integration (see "Key concepts" below).
4. **Economic-Advisor Agent** — this is the actual novel piece. It combines
   the diagnosis, the risk level, and the market price into a genuine
   cost-benefit calculation: expected value of yield saved by treating,
   versus the cost of treatment, adjusted for how close harvest is. It
   outputs one of: spray, spray (marginal benefit — consider a cheaper
   input), don't spray, or don't spray (too close to harvest — check the
   pre-harvest interval).
5. **Action Agent** — only engages if the recommendation is to spray and
   the farmer wants to act on it. It drafts an order message to a local
   input dealer, and only *sends* it after an explicit human confirmation
   step. It can also schedule a reapplication reminder.

See `docs/architecture.svg` in the repository for the full diagram.

The orchestrator is a standard ADK `Agent` with `sub_agents=[...]`; it uses
ADK's built-in delegation (`transfer_to_agent`) to route a farmer's message
to the right specialist in sequence, rather than a hand-coded if/else
pipeline. This means the conversation stays natural — a farmer can ask a
follow-up question and the orchestrator decides which specialist should
answer it, instead of forcing a rigid form-fill flow.

## Key concepts demonstrated

| Concept | Where | Notes |
|---|---|---|
| Multi-agent system (ADK) | `agents.py` | 5 specialist `Agent`s + 1 orchestrator with `sub_agents`, verified against the real installed `google-adk==2.3.0` API (not written from memory — the graph builds and its structure was tested). |
| Security features | `agents.py`, `tools.py` | `send_dealer_order` and `schedule_reapplication_reminder` are registered with `require_confirmation=True`. ADK will not execute either without an explicit human "yes" — the agent physically cannot place an order or send a message unattended. |
| MCP Server | `market_intel_agent` in `agents.py` | Currently a local Python stub function so the pipeline is runnable end-to-end without external credentials; the README documents exactly how to swap it for a real `MCPToolset` connection to a market-price MCP server. Being transparent about this rather than faking a live MCP connection for the demo. |
| Agent skills / Antigravity / Deployability | Video | Demonstrated live in the walkthrough video rather than in static code — see video for the Antigravity build session and deployment discussion. |

## Technical implementation notes

Two functions in `tools.py` carry the real intelligence of the system and
are worth calling out explicitly for judges:

- `compute_disease_risk()` — a simplified but genuine humidity/temperature
  heuristic for fungal disease spread risk.
- `compute_economics()` — the actual spray/don't-spray decision math:
  `expected_loss_value = expected_yield_kg × loss_pct × price_per_kg`,
  compared against treatment cost, with a hard rule that treatment this
  close to harvest (≤7 days) triggers a pre-harvest-interval warning
  instead of a spray recommendation.

Everything else that would require a live external API — weather data and
market price — is implemented as a clearly labeled stub (the return value
literally says `"STUB DATA"` or `"SIMULATED"`) so that anyone reading the
code, or a judge running it, knows exactly which numbers are real
computation and which are placeholders waiting on a real data source. I
chose to be explicit about this rather than paper over it, because a
crop-economics tool that quietly guesses at prices is worse than one that
says plainly "this number isn't real yet."

## Real-world value and path to income

This isn't designed to stop at the competition. Three concrete monetization
paths exist without inventing new business models:

- **Dealer commission** — local agri-input dealers pay a small referral fee
  for orders routed to them through the Action Agent.
- **FPO/cooperative subscription** — Farmer Producer Organizations pay a
  flat fee for their member base to use the system, instead of per-farmer
  pricing that smallholders can't absorb.
- **B2B data licensing** — anonymized, consented crop-health and
  spray-decision data has real value to agri-input companies and crop
  insurers for underwriting, a market that already exists today.

## Honest limitations

- The disease-risk and economics heuristics need review by an agronomist
  per crop before the numbers are trusted for real farm decisions.
- Diagnosis accuracy on real-world phone photos (bad lighting, blur, wrong
  crop part) hasn't been validated against a labeled photo set yet.
- The market-price and weather integrations are stubs, clearly labeled as
  such, pending a real API or MCP server connection.
- No real payment/ordering flow exists — `send_dealer_order` is simulated.
  Wiring an actual dealer network in is the largest remaining piece of
  work, and it's a partnerships problem as much as an engineering one.

## Why this, and not another diagnosis app

The honest pitch here isn't "we built a better plant-disease classifier" —
diagnosis-only tools already exist and do that job reasonably well. The
contribution is the layer nobody has built yet: an agent that takes that
diagnosis and asks the question a farmer actually cares about — *is this
worth spending money on, right now, given my harvest date and today's
price* — and only then, with explicit permission, does something about it.
That's a genuinely different product, built the way multi-agent systems
are supposed to be used: specialists that each do one job well, coordinated
by an orchestrator, gated by a human at the one point where real money
moves.


---
## 7. Video Script (5-minute walkthrough)

# AgriMargin — Video Script (target: 4:30–5:00 total)

Record this as a screen-share with voiceover. Suggested tools: your phone
or OBS/Loom for recording, Antigravity IDE or your terminal for the demo
segment. Publish unlisted-or-public to YouTube (must be public, not
unlisted, per the rubric — double check before submitting).

---

### 0:00–0:40 — Problem Statement

**Say:**
"A farmer with a sick tomato plant today has to piece together information
from three different places. A photo-ID app tells them what disease it
might be. A weather app tells them nothing about disease risk. And a
separate trip to the market tells them the price — days later. Nothing
puts those together and answers the question that actually determines
their income: is treating this worth the money, given how close I am to
harvest? That gap costs real farmers real money — either they over-spend
on treatment that doesn't pay for itself, or they under-treat a genuinely
dangerous outbreak because nobody ran the numbers."

**Show:** a simple title card or the problem stated as text on screen.

---

### 0:40–1:20 — Why Agents

**Say:**
"This isn't one classification problem, it's a small research-and-decide
workflow — plant pathology, meteorology, and market economics all have to
be reasoned over together, and then, only with permission, acted on. That's
exactly what multi-agent orchestration is for: specialists that each do one
job well, coordinated by an orchestrator, with a hard stop for human
approval before anything real happens."

**Show:** `docs/architecture.svg` on screen while you talk.

---

### 1:20–2:20 — Architecture

**Say, walking through the diagram left to right:**
"The farmer's question goes to the orchestrator — a standard ADK agent
that delegates to five specialists using ADK's built-in agent-transfer
mechanism, not a hardcoded pipeline. The Diagnosis Agent identifies the
disease from a photo. The Weather-Risk Agent computes a real
humidity/temperature disease-spread risk score. The Market-Intel Agent
gets the current price — this is the natural slot for a real MCP server,
which I'll show in a moment. The Economic-Advisor Agent is the actual
novel piece: it combines all three into a genuine cost-benefit calculation
— expected value of yield saved, versus treatment cost, adjusted for days
to harvest. And the Action Agent only fires if the recommendation is to
spray, and it cannot send anything — an order or a message — without an
explicit human confirmation step. That's enforced by ADK's
`require_confirmation` flag on the tool itself, not just a prompt asking
the model nicely."

**Show:** brief cut to `agents.py`, highlighting the `sub_agents=[...]`
line and the `require_confirmation=True` line.

---

### 2:20–3:50 — Demo

**Say while running it:**
"Here it is running end to end."

**Do:**
1. Run `python main.py --demo` (or interactive mode with a live typed
   example) and narrate the output as each agent responds.
2. Show the Economic-Advisor's recommendation and the actual numbers
   (expected loss value, treatment cost, net benefit) — point out that
   this is real arithmetic, not a made-up answer.
3. Show the Action Agent pausing for confirmation before drafting/sending
   the dealer order — this is the moment to emphasize the safety rail.
4. **[If you swapped the market-price stub for a real MCP server before
   recording]** show that call happening live and say so explicitly:
   "This price is coming from a live MCP server connected to [source],
   not the stub."

**If you haven't swapped in a real MCP/weather source yet**, say so
honestly on camera: "Right now the weather and price data are clearly
labeled stub functions so the pipeline runs end-to-end without external
credentials — the README documents exactly how to swap them for a real
market-data MCP server."

---

### 3:50–4:30 — The Build

**Say:**
"This was built on Google's Agent Development Kit — `agents.py` defines
five specialist agents plus an orchestrator using ADK's `Agent` and
`sub_agents`, verified against the actual installed ADK 2.3.0 API. [If
applicable:] I used Google Antigravity's agent-first IDE to iterate on
this — here's a quick look at that workflow [cut to a short clip of
Antigravity's Mission Control / agent panel making an edit or running a
task]. For deployment, this is built to run as a standard ADK app, which
can be deployed via Vertex AI Agent Engine or as a Managed Agent through
the Gemini API — I haven't deployed it live for judging, per the rubric,
but the path is documented in the README."

**Show:** Antigravity IDE session if you have one recorded; otherwise a
quick terminal walkthrough of `pip install -r requirements.txt` and
`python main.py`.

---

### 4:30–5:00 — Close

**Say:**
"The honest pitch here isn't a better plant-disease classifier — those
already exist. It's the layer nobody's built yet: an agent that takes a
diagnosis and asks the question a farmer actually cares about — is this
worth spending money on right now — and only then, with permission, acts
on it. Code and full writeup are linked below."

**Show:** GitHub repo URL and Kaggle Writeup link as an end card.

---

## Recording checklist before you hit publish

- [ ] Video is 5:00 or under
- [ ] Published to YouTube and set to **Public** (not private/unlisted —
      confirm the rubric's exact requirement before submitting)
- [ ] Covers all five required beats: Problem, Agents, Architecture, Demo,
      Build
- [ ] If you didn't get to a real MCP/Antigravity integration, the script
      above still gets you full marks on the *pitch* criteria — just be
      honest on camera about what's real vs. stubbed, as in the Writeup


---
## 8. Submission Checklist

# Submission Checklist — AI Agents Capstone (due Jul 7, 2026, 9:59 AM GMT+3)

Use this as your final pass. Items are ordered by priority given the
deadline — do the "Must do" section first, everything else is polish.

## Required submission components

| Requirement | Status | Action |
|---|---|---|
| Kaggle Writeup (title, subtitle, analysis, ≤2500 words) | ✅ Drafted (`docs/KAGGLE_WRITEUP.md`, ~1,300 words) | Paste into a new Kaggle Writeup, select your Track (**Agents for Good**), attach the required assets below |
| Cover image (Media Gallery) | ❌ Not created | Take a clean screenshot of `docs/architecture.svg` or the demo running — needs to be visually appealing, this is your cover image |
| Video (≤5 min, published to YouTube) | ❌ Not recorded | Use `docs/VIDEO_SCRIPT.md`. **Must be public on YouTube.** |
| Public Project Link (working demo, or GitHub repo + setup instructions if no live demo) | ⚠️ Code exists, not yet on GitHub | Push this project to a **public** GitHub repo (see below) |

## Must do before submitting (in order)

1. **Push to a public GitHub repo.** You almost certainly don't have a
   live public demo endpoint given the timeline, so the GitHub repo *is*
   your project link. Make sure it's public, not private.
   ```bash
   cd agrimargin
   git init
   git add .
   git commit -m "AgriMargin capstone submission"
   # create a public repo on github.com, then:
   git remote add origin https://github.com/<you>/agrimargin.git
   git branch -M main
   git push -u origin main
   ```
2. **Double-check no API key is committed.** Run `git status` and confirm
   `.env` (if you created one) is not staged — only `.env.example` should
   be. The rubric explicitly flags this as disqualifying if violated.
3. **Record the video** using `docs/VIDEO_SCRIPT.md`. Even without a real
   MCP/Antigravity integration, the honest version of this script still
   covers all five graded beats (Problem, Agents, Architecture, Demo,
   Build).
4. **Create the Kaggle Writeup**, paste in `docs/KAGGLE_WRITEUP.md`,
   attach cover image + video, select Track = Agents for Good, submit.

## Key concepts — where you stand (need at least 3 of 6)

| Concept | Status | Where it's demonstrated |
|---|---|---|
| Agent/multi-agent system (ADK) | ✅ Done | `agents.py` — 5 specialists + orchestrator, real `google-adk` API |
| Security features | ✅ Done | `agents.py`/`tools.py` — `require_confirmation=True` on side-effecting tools |
| MCP Server | ⚠️ Stub, documented path to real | `market_intel_agent` — swap instructions in `README.md`. If you have time, this is the single highest-value thing left to do — even one real MCP call strengthens the submission a lot. |
| Deployability | 🎤 Video only | Mention in the "Build" section of the video — Vertex AI Agent Engine or Gemini API Managed Agents are the natural targets. You are not required to actually deploy. |
| Agent skills / Antigravity | 🎤 Video only | If you have Antigravity installed, do a short visible iteration on camera (e.g. open the repo in Antigravity, use its agent panel to make one small edit, or define a skill.md). If you don't have time, it's fine to skip this row — you already have 3 concepts covered from the table above (multi-agent, security, and either MCP or the two video-only rows), which meets the minimum. |

You already clear the "at least 3 of 6" bar with multi-agent system +
security features + MCP (even as a documented stub with real integration
instructions, this is defensible) — everything past that is upside, not
a requirement.

## Documentation (20 of 70 implementation points)

`README.md` already covers: problem, solution, architecture (with
`docs/architecture.svg`), setup instructions, and an honest real-vs-stub
table. Before submitting, re-read it once as if you were a judge seeing
this cold — that's the single best use of your remaining review time.

## Nice-to-have if you have spare hours

- Real weather API swap in `get_weather_forecast()` — lower effort than
  MCP, still removes one "STUB" label from the demo.
- A short labeled-photo test of the Diagnosis Agent against 5-10 real
  crop images, with results noted in the Writeup's limitations section —
  turns a stated limitation into "tested and here's what we found."
